In [78]:
import requests
import os
import tvdb_v4_official as apitv
import re 

APIKEY = "3f9cbd45-f38b-463c-8d97-89e9d6ed94ea"
tvdb = apitv.TVDB(APIKEY)


In [79]:
def getTVDBToken(token):
    url = 'https://api4.thetvdb.com/v4/login'
    headers = {
        'Content-Type': 'application/json'
    }
    payload = {
        'apikey': token
    }
    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    return response.json()['data']['token']


def findSeriesByName(tvdb: apitv.TVDB, name: str, token):
    try:

        anime = tvdb.get_series_by_slug(_setFormatSlug(name))
        
        return anime
    except ValueError as e:
        return searchAnime(tvdb, name)
    
def searchAnime(tvdb: apitv.TVDB, name):

    tvdb.search(name)

    return tvdb.search(name)


def _setFormatSlug(text):
    text = re.sub(r'-1$', '', text)
    slugText = text.replace(" ", "-").lower()
    return slugText


In [80]:
episodes = tvdb.get_season_extended(seasons[6]["id"])["episodes"]
[episode["name"] for episode in episodes]

In [81]:
tvdb.get_all_seasons()

In [82]:
token = getTVDBToken(APIKEY)
name = 'Breaking bad'
detalles = findSeriesByName(tvdb, name, token)
if detalles:
    print(detalles)
else:
    print("No se encontraron detalles para la serie especificada.")

In [83]:
allSeries = tvdb.get_all_series(page = 1)
name = 'Dragon Ball Super 2015'

In [84]:
search = tvdb.search(name)
search[0].keys()

In [172]:
search

[{'objectID': 'movie-8645',
  'aliases': ['Fairy Tail : Dragon Cry (Original Japanese Version)'],
  'country': 'jpn',
  'director': 'Tatsuma Minamikawa',
  'extended_title': 'Fairy Tail: The Movie - Dragon Cry (2017)',
  'genres': ['Action', 'Adventure', 'Comedy', 'Fantasy', 'Animation'],
  'id': 'movie-8645',
  'image_url': 'https://artworks.thetvdb.com/banners/movies/8645/posters/8645.jpg',
  'name': 'Fairy Tail: The Movie - Dragon Cry',
  'first_air_time': '2017-05-06',
  'overview': "In the Kingdom of Fiore, Zash Caine, the Kingdom of Stella's minister of state and a practitioner of black magic, steals the Dragon Cry, a mystical staff discovered in the dragon graveyard beneath the capital city of Crocus. Fiore's royal family recruits Natsu Dragneel and his team from the Fairy Tail guild to recover the staff, which is imbued with magical power capable of annihilating the kingdom. The wizards pursue Zash to Stella, whose ruler, King Animus, intends to use the staff for a ritual. Nats

In [86]:
seasons = tvdb.get_series_extended(search[0]["id"][7:])["seasons"]
season = tvdb.get_season_extended(seasons[0]["id"])['episodes']
number = len(season)

print(number)

In [87]:
from datetime import datetime

def getAired(episode):
    if not episode.get('aired'):
        return None
    try:
        return datetime.fromisoformat(episode['aired'])
    except ValueError:
        return None


In [88]:
def animeSearched(animeTitle, animeYear):
    search = tvdb.search_series(
        name=animeTitle.lower(),  # Título del anime
        firstAired=animeYear  # Año de emisión
    )
    if not search.get("data"):
        raise ValueError(f"No se encontraron resultados para '{animeTitle}' en {animeYear}.")
    return search["data"]


In [89]:

def getSeasonsDates(animeTitle, animeYear):
    try:
        seasons = tvdb.get_series_extended(animeSearched(animeTitle, animeYear)[0]["id"][7:])["seasons"]
        seasonDates = []
        for season in seasons:
            if season['number'] not in [0, 1, len(season)-1]:
                dates = [getAired(episode) for episode in tvdb.get_season_extended(season['id'])['episodes']]
                dates.sort()

                seasonDates.append({
                    "Temporada": season,
                    "StartDate": dates[0],
                    "EndDate": dates[-1]
                })
        return seasonDates
    except Exception as e:
        return None

In [90]:
def getAnimeListAllEpisodes(animeTitle, animeYear):
    try:
        seasons = tvdb.get_series_extended(animeSearched(animeTitle, animeYear)[0]["id"][7:])["seasons"]   
        episodesForSeasons = []
        for season in seasons:
            if season['type']['name'] == 'Aired Order':
                if season['number'] not in [0, len(season)-1]:
                    episodes =  len(tvdb.get_season_extended(season['id'])['episodes'])
                    episodesForSeasons.append(episodes)
        
        return episodesForSeasons
    except Exception as e:
        return None

In [91]:
def getSeasonsNumTVDB(animeTitle, animeYear):
    try:
        series_id = tvdb.get_series_extended(animeSearched(animeTitle, animeYear)[0]["id"][7:])["seasons"]
        seasonsNumberList = []

        for season in series_id:  
            if season.get('type', {}).get('name', '').lower() == 'aired order' and season['number'] not in [0, len(series_id) - 1]:
                seasonsNumberList.append(season['number'])  
            
        return seasonsNumberList
    except Exception as e:
        return None

# Prueba funciones 

In [199]:
def getSeasonsDates1(animeTitle, animeYear):
    try:
        search = tvdb.search(animeTitle)
        search = [s for s in search if s['type'] == "series" and s['year'] == str(animeYear)]
        seasons = tvdb.get_series_extended(search[0]["tvdb_id"])["seasons"]
        seasonDates = []
        official_seasons = [season for season in seasons if season['type']['type'] == 'official' and season['number'] != 0]

        for season in official_seasons:
            print(season)
            dates = [getAired(episode) for episode in tvdb.get_season_extended(season['id'])['episodes']]
            dates.sort()
            seasonDates.append({
            "Temporada": season['number'],
            "StartDate": dates[0],
            "EndDate": dates[-1]
            })
        return seasonDates
    except Exception as e:
        return print("Error en la búsqueda")

In [ ]:
animeTitle = "Fairy Tail"
animeYear = 2009

search = tvdb.search(animeTitle)
for s in search:
    if s['type'] == "series":
        if 'year' in s.keys():

            print(s)
#search = [s for s in search if s['type'] == "series" and s['year'] == str(animeYear)]





"""
search = [s for s in search if s['type'] == "series" and s['year'] == str(animeYear)]
seasons = tvdb.get_series_extended(search[0]["tvdb_id"])["seasons"]
seasonDates = []
official_seasons = [season for season in seasons if season['type']['type'] == 'official' and season['number'] != 0]

for season in official_seasons:
    print(season)
    dates = [getAired(episode) for episode in tvdb.get_season_extended(season['id'])['episodes']]
    dates.sort()
    seasonDates.append({
    "Temporada": season['number'],
    "StartDate": dates[0],
    "EndDate": dates[-1]
    })
seasonDates
"""



True
{'objectID': 'series-410031', 'aliases': ['Fairy Tail - 100 Years Quest', 'FAIRY TAIL: 100 Years Quest', 'Fairy Tail - La Quête de Cent Ans', 'Fairy Tail 100 Years Quest : En route pour la quête de 100 ans', 'Fairy Tail - 100 Years Quest', '百年任务', 'Fairy Tail: Quest dos 100 Anos', 'Fairy Tail: Quest dos 100 Anos', 'Fairy Tail: Nhiệm Vụ 100 Năm', 'Fairy Tail Nhiệm Vụ 100 Năm', 'Hội Pháp Sư: Nhiệm Vụ 100 Năm', 'Hội Pháp Sư Nhiệm Vụ 100 Năm', 'Nhiệm Vụ 100 Năm', 'فيري تيل: مهمة المئة عام', 'ذيل الجنية: مهمة المئة عام'], 'country': 'jpn', 'id': 'series-410031', 'image_url': 'https://artworks.thetvdb.com/banners/v4/series/410031/posters/670bc84e7a555.jpg', 'name': 'FAIRY TAIL 100 年クエスト', 'first_air_time': '2024-07-07', 'overview': 'ネーム原作/真島ヒロ 作画/上田敦夫\r\nやっぱり『FAIRY TAIL』は終わってなんかいなかった！ ゼレフ、アクノロギアとの死闘を乗り越え、さらに強さと騒々しさを増した“妖精の尻尾”！ ＜100年クエスト＞に出かけたナツたちの気になる旅路に、ギルドに残ったメンバーにも何やら新しいことが起きそうで…!? 完結したはずの545話目から、そのままつづく新たなる物語！', 'primary_language': 'jpn', 'primary_type': 'series', 'status': 'Continu

'\nsearch = [s for s in search if s[\'type\'] == "series" and s[\'year\'] == str(animeYear)]\nseasons = tvdb.get_series_extended(search[0]["tvdb_id"])["seasons"]\nseasonDates = []\nofficial_seasons = [season for season in seasons if season[\'type\'][\'type\'] == \'official\' and season[\'number\'] != 0]\n\nfor season in official_seasons:\n    print(season)\n    dates = [getAired(episode) for episode in tvdb.get_season_extended(season[\'id\'])[\'episodes\']]\n    dates.sort()\n    seasonDates.append({\n    "Temporada": season[\'number\'],\n    "StartDate": dates[0],\n    "EndDate": dates[-1]\n    })\nseasonDates\n'

In [151]:
getSeasonsDates1("Dragon ball super", 2015)

{'id': 624509, 'seriesId': 295068, 'type': {'id': 1, 'name': 'Aired Order', 'type': 'official', 'alternateName': None}, 'name': '破壊神ビルス編', 'number': 1, 'nameTranslations': ['deu,eng,fra,ita,jpn,kor,pt,spa,zho'], 'overviewTranslations': ['deu,eng,fra,ita,kor,pt,spa,zho'], 'image': 'https://artworks.thetvdb.com/banners/v4/season/624509/posters/607400e703d6e.jpg', 'imageType': 7, 'companies': {'studio': None, 'network': None, 'production': None, 'distributor': None, 'special_effects': None}, 'lastUpdated': '2025-01-09 11:51:30'}
{'id': 644129, 'seriesId': 295068, 'type': {'id': 1, 'name': 'Aired Order', 'type': 'official', 'alternateName': None}, 'name': 'フリーザ復活編', 'number': 2, 'nameTranslations': ['deu,eng,fra,ita,jpn,kor,pt,spa,zho'], 'overviewTranslations': ['deu,eng,fra,ita,kor,spa,zho'], 'image': 'https://artworks.thetvdb.com/banners/v4/season/644129/posters/6074013d7bf70.jpg', 'imageType': 7, 'companies': {'studio': None, 'network': None, 'production': None, 'distributor': None, 'sp

[{'Temporada': 1,
  'StartDate': datetime.datetime(2015, 7, 5, 0, 0),
  'EndDate': datetime.datetime(2015, 10, 11, 0, 0)},
 {'Temporada': 2,
  'StartDate': datetime.datetime(2015, 10, 18, 0, 0),
  'EndDate': datetime.datetime(2016, 1, 17, 0, 0)},
 {'Temporada': 3,
  'StartDate': datetime.datetime(2016, 1, 24, 0, 0),
  'EndDate': datetime.datetime(2016, 6, 5, 0, 0)},
 {'Temporada': 4,
  'StartDate': datetime.datetime(2016, 6, 12, 0, 0),
  'EndDate': datetime.datetime(2017, 1, 29, 0, 0)},
 {'Temporada': 5,
  'StartDate': datetime.datetime(2017, 2, 5, 0, 0),
  'EndDate': datetime.datetime(2018, 3, 25, 0, 0)}]

In [159]:
getSeasonsDates1("My Hero Academia", 2016)

{'id': 651039, 'seriesId': 305074, 'type': {'id': 1, 'name': 'Aired Order', 'type': 'official', 'alternateName': None}, 'number': 1, 'nameTranslations': [], 'overviewTranslations': ['ara,eng,ita,por'], 'image': 'https://artworks.thetvdb.com/banners/v4/season/651039/posters/63596c6287b92.jpg', 'imageType': 7, 'companies': {'studio': None, 'network': None, 'production': None, 'distributor': None, 'special_effects': None}, 'lastUpdated': '2024-10-20 23:07:28'}
{'id': 672237, 'seriesId': 305074, 'type': {'id': 1, 'name': 'Aired Order', 'type': 'official', 'alternateName': None}, 'number': 2, 'nameTranslations': [], 'overviewTranslations': ['ara,eng,fra,ita,por,pt'], 'image': 'https://artworks.thetvdb.com/banners/v4/season/672237/posters/63596b9c82c6f.jpg', 'imageType': 7, 'companies': {'studio': None, 'network': None, 'production': None, 'distributor': None, 'special_effects': None}, 'lastUpdated': '2024-10-21 02:00:06'}
{'id': 732155, 'seriesId': 305074, 'type': {'id': 1, 'name': 'Aired O

[{'Temporada': 1,
  'StartDate': datetime.datetime(2016, 4, 3, 0, 0),
  'EndDate': datetime.datetime(2016, 6, 26, 0, 0)},
 {'Temporada': 2,
  'StartDate': datetime.datetime(2017, 4, 1, 0, 0),
  'EndDate': datetime.datetime(2017, 9, 30, 0, 0)},
 {'Temporada': 3,
  'StartDate': datetime.datetime(2018, 4, 7, 0, 0),
  'EndDate': datetime.datetime(2018, 9, 29, 0, 0)},
 {'Temporada': 4,
  'StartDate': datetime.datetime(2019, 10, 12, 0, 0),
  'EndDate': datetime.datetime(2020, 4, 4, 0, 0)},
 {'Temporada': 5,
  'StartDate': datetime.datetime(2021, 3, 27, 0, 0),
  'EndDate': datetime.datetime(2021, 9, 25, 0, 0)},
 {'Temporada': 6,
  'StartDate': datetime.datetime(2022, 10, 1, 0, 0),
  'EndDate': datetime.datetime(2023, 3, 25, 0, 0)},
 {'Temporada': 7,
  'StartDate': datetime.datetime(2024, 5, 4, 0, 0),
  'EndDate': datetime.datetime(2024, 10, 12, 0, 0)}]

In [153]:
getSeasonsDates1('Fairy Tail', 2009)

Error en la búsqueda


In [45]:
getAnimeListAllEpisodes('Fairy Tail', 2009)

In [95]:
getSeasonsNumTVDB('Fairy tail', 2009)